In [209]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import os
from dotenv import load_dotenv
import springernature_api_client.meta as meta

#### Getting the data

In [210]:
def load_with_source(folder_path, source_name):
    folder = Path(folder_path)
    dfs = []
    for file in folder.glob("*.csv"):
        df = pd.read_csv(file)
        df["source"] = source_name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [211]:
# getting the data
acm = load_with_source("data/raw/ACM", "ACM")
ieee = load_with_source("data/raw/IEEE", "IEEE")
springer = load_with_source("data/raw/Springer", "Springer")

sd = pd.read_csv("data/raw/ScienceDirect/ScienceDirect_query01.csv")
sd["source"] = "ScienceDirect"

In [212]:
# create table to keep track of counts
count_columns = ["data", "ACM", "IEEE", "Springer", "ScienceDirect"]

count_db = pd.DataFrame(columns = count_columns)
count_db.loc[0] = ["raw", len(acm), len(ieee), len(springer), len(sd)]
count_db

,data,ACM,IEEE,Springer,ScienceDirect
0,raw,78,337,1499,49


In [213]:
# getting column names for all DB
acm_col = list(acm)
ieee_col = list(ieee)
springer_col = list(springer)
sd_col = list(sd)

In [214]:
print("acm", acm_col)
print("ieee", ieee_col)
print("springer", springer_col)
print("sd", sd_col)

acm ['Key', 'Item Type', 'Publication Year', 'Author', 'Title', 'Publication Title', 'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract Note', 'Date', 'Date Added', 'Date Modified', 'Access Date', 'Pages', 'Num Pages', 'Issue', 'Volume', 'Number Of Volumes', 'Journal Abbreviation', 'Short Title', 'Series', 'Series Number', 'Series Text', 'Series Title', 'Publisher', 'Place', 'Language', 'Rights', 'Type', 'Archive', 'Archive Location', 'Library Catalog', 'Call Number', 'Extra', 'Notes', 'File Attachments', 'Link Attachments', 'Manual Tags', 'Automatic Tags', 'Editor', 'Series Editor', 'Translator', 'Contributor', 'Attorney Agent', 'Book Author', 'Cast Member', 'Commenter', 'Composer', 'Cosponsor', 'Counsel', 'Interviewer', 'Producer', 'Recipient', 'Reviewed Author', 'Scriptwriter', 'Words By', 'Guest', 'Number', 'Edition', 'Running Time', 'Scale', 'Medium', 'Artwork Size', 'Filing Date', 'Application Number', 'Assignee', 'Issuing Authority', 'Country', 'Meeting Name', 'Conference Name', 'Court', '

### prepare the individual DB so they can be merged into one later

##### we are with making sure all db only have for journal articles and conference papers / proceedings

In [215]:
# create function for filter for journal articles and conferences papers 
def filter_journal_conference(df, column_name):
    '''
    This funciton...
    '''
    # define regex pattern
    pattern = re.compile(r"conference|journal|article", re.IGNORECASE)

    # ensure all entries are strings and strip whitespace
    cleaned_col = df[column_name].astype(str).str.strip()

    # filter for rows matching the pattern
    filtered_df = df[cleaned_col.str.contains(pattern, na = False)]

    return filtered_df

In [216]:
acm_type_filtered = filter_journal_conference(acm, "Item Type")
ieee_type_filtered = filter_journal_conference(ieee, "Document Identifier")
springer_type_filtered = filter_journal_conference(springer, "Content Type")
sd_type_filtered = filter_journal_conference(sd, "Item Type")

In [217]:
# update counting DB
type_filter_row = pd.DataFrame([["type filtered", len(acm_type_filtered), len(ieee_type_filtered),
                            len(springer_type_filtered), len(sd_type_filtered)]],
                            columns = count_db.columns)

count_db = pd.concat([count_db, type_filter_row], ignore_index = True)

count_db

,data,ACM,IEEE,Springer,ScienceDirect
0,raw,78,337,1499,49
1,type filtered,76,337,1499,49


##### next, filter for entires containing "fall detection" AND ("privacy" OR "security") in title OR abstract

In [218]:
# create function for filter for journal articles and conferences papers 
def filter_title_abstract(df, column_names):
    '''
    This funciton...
    '''
    # define regex pattern
    pattern = re.compile(r"fall detection" and r"security|privacy", re.IGNORECASE)

    for col in column_names:
        # ensure all entries are strings and strip whitespace
        cleaned_col = df[col].astype(str).str.strip()

        # filter for rows matching the pattern
        filtered_df = df[cleaned_col.str.contains(pattern, na = False)]

    return filtered_df

In [219]:
print("acm", acm_col)
print("ieee", ieee_col)
print("springer", springer_col)
print("sd", sd_col)

acm ['Key', 'Item Type', 'Publication Year', 'Author', 'Title', 'Publication Title', 'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract Note', 'Date', 'Date Added', 'Date Modified', 'Access Date', 'Pages', 'Num Pages', 'Issue', 'Volume', 'Number Of Volumes', 'Journal Abbreviation', 'Short Title', 'Series', 'Series Number', 'Series Text', 'Series Title', 'Publisher', 'Place', 'Language', 'Rights', 'Type', 'Archive', 'Archive Location', 'Library Catalog', 'Call Number', 'Extra', 'Notes', 'File Attachments', 'Link Attachments', 'Manual Tags', 'Automatic Tags', 'Editor', 'Series Editor', 'Translator', 'Contributor', 'Attorney Agent', 'Book Author', 'Cast Member', 'Commenter', 'Composer', 'Cosponsor', 'Counsel', 'Interviewer', 'Producer', 'Recipient', 'Reviewed Author', 'Scriptwriter', 'Words By', 'Guest', 'Number', 'Edition', 'Running Time', 'Scale', 'Medium', 'Artwork Size', 'Filing Date', 'Application Number', 'Assignee', 'Issuing Authority', 'Country', 'Meeting Name', 'Conference Name', 'Court', '

In [220]:
# Load .env file
load_dotenv()

# Access the API key
SPRINGER_API_KEY = os.getenv("SPRINGER_API_KEY")

if not SPRINGER_API_KEY:
    raise ValueError("Springer API key not found. Make sure .env exists and contains SPRINGER_API_KEY")

In [ ]:
# initialize API Client
meta_client = meta.MetaAPI(api_key = SPRINGER_API_KEY) 

# abstract, keyword, startingPage, endingPage

In [233]:
acm

,Key,Item Type,Publication Year,Author,Title,Publication Title,ISBN,ISSN,DOI,Url,...,Version,System,Code,Code Number,Section,Session,Committee,History,Legislative Body,source
0,MTGEZS9Z,journalArticle,2025,"Meng, Chengzhen; He, Chenming; Wang, Dequan; X...",GR-Fall: A Fall Detection System with Gait Rec...,Proc. ACM Interact. Mob. Wearable Ubiquitous T...,NaN,NaN,10.1145/3749471,https://doi.org/10.1145/3749471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
1,J3DCVG26,conferencePaper,2025,"Zhang, Xinyu; Wang, Jia; Ma, Sen; Gong, Tailong",Study of fall detection in infrared scenes bas...,Proceedings of the 4th Asia-Pacific Artificial...,979-8-4007-1086-5,NaN,10.1145/3718491.3718627,https://doi.org/10.1145/3718491.3718627,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
2,K2P8FBHV,conferencePaper,2024,"Correia, Sérgio D.; Matos-Carvalho, João P.; T...",Quantization with Gate Disclosure for Embedded...,Proceedings of the 2024 International Conferen...,979-8-4007-1094-0,NaN,10.1145/3677525.3678644,https://doi.org/10.1145/3677525.3678644,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
3,UJ5EX5EG,conferencePaper,2025,"Li, Lin; Jiang, Lei; Han, Han; Yuan, Keya",Smart Floor System for Comprehensive Fall Dete...,Proceedings of the 2024 5th International Symp...,979-8-4007-1782-6,NaN,10.1145/3706890.3706935,https://doi.org/10.1145/3706890.3706935,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
4,2ICWWWB6,conferencePaper,2025,"Zhou, Manyue; Gong, Pengcheng; Wu, Yuntao; Liu...",Point Cloud Classification Fall Detection Meth...,Proceedings of the 2024 4th International Join...,979-8-4007-1010-0,NaN,10.1145/3696474.3696488,https://doi.org/10.1145/3696474.3696488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,8DBL22KX,conferencePaper,2024,"Prasad, Shyam Sunder; Mehta, Naval Kishore; Ku...",Hybrid SNN-based Privacy-Preserving Fall Detec...,Proceedings of the Fourteenth Indian Conferenc...,979-8-4007-1625-6,NaN,10.1145/3627631.3627650,https://doi.org/10.1145/3627631.3627650,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
74,F2T9WAIA,conferencePaper,2023,"Zhang, Runqing; Yan, Haoran; Cai, Dunbo; Qian,...",Black Cloud Generation Method for Privacy Prot...,Proceedings of the 2022 6th International Conf...,978-1-4503-9756-8,NaN,10.1145/3579109.3579112,https://doi.org/10.1145/3579109.3579112,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
75,RELEWXRY,conferencePaper,2019,"Mainali, Pradip; Shepherd, Carlton",Privacy-Enhancing Fall Detection from Remote S...,Proceedings of the 14th International Conferen...,978-1-4503-7164-3,NaN,10.1145/3339252.3340500,https://doi.org/10.1145/3339252.3340500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM
76,4HLI79TR,conferencePaper,2022,"Chen, Zhiwen; Ji, Yong; Ren, Yuanhong",Research on the Application of Fall Detection ...,Proceedings of the 2022 3rd International Conf...,978-1-4503-9548-9,NaN,10.1145/3512826.3512832,https://doi.org/10.1145/3512826.3512832,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACM


In [231]:
databases = {"ACM": acm, "IEEE": ieee, "Springer": springer, "SD": sd}

for name, df in databases.items():
    print(f"\n{name} column types:")
    print(df.dtypes)


ACM column types:
Key                     str
Item Type               str
Publication Year      int64
Author                  str
Title                   str
                     ...   
Session             float64
Committee           float64
History             float64
Legislative Body    float64
source                  str
Length: 88, dtype: object

IEEE column types:
Document Title                str
Authors                       str
Author Affiliations           str
Publication Title             str
Date Added To Xplore          str
Publication Year            int64
Volume                     object
Issue                     float64
Start Page                 object
End Page                   object
Abstract                      str
ISSN                          str
ISBNs                         str
DOI                           str
Funding Information           str
PDF Link                      str
Author Keywords               str
IEEE Terms                    str
Mesh_Terms     

2026-03-02 18:17:14,971 - INFO - Making request to: https://api.springernature.com/meta/v2/json with params: {'q': 'doi:"10.1007/s10489-024-05645-1"', 'p': 10, 's': 1, 'api_key': '9a7f05d813a72418537730d30f9d874c'}


Fetching meta: query='doi:"10.1007/s10489-024-05645-1"', page=10, start=1


{'records': [{'contentType': 'Article',
   'identifier': 'doi:10.1007/s10489-024-05645-1',
   'language': 'en',
   'url': [{'format': 'html',
     'platform': 'web',
     'value': 'http://link.springer.com/openurl/fulltext?id=doi:10.1007/s10489-024-05645-1'},
    {'format': 'pdf',
     'platform': 'web',
     'value': 'http://link.springer.com/openurl/pdf?id=doi:10.1007/s10489-024-05645-1'},
    {'format': '',
     'platform': '',
     'value': 'http://dx.doi.org/10.1007/s10489-024-05645-1'}],
   'title': 'Deep learning for computer vision based activity recognition and fall detection of the elderly: a systematic review',
   'creators': [{'creator': 'Gaya-Morey, F. Xavier',
     'ORCID': '0000-0003-1231-7235'},
    {'creator': 'Manresa-Yee, Cristina', 'ORCID': '0000-0002-8482-7552'},
    {'creator': 'Buades-Rubio, José M.', 'ORCID': '0000-0002-6137-9558'}],
   'publicationName': 'Applied Intelligence',
   'openaccess': 'true',
   'doi': '10.1007/s10489-024-05645-1',
   'publisher': 'Sp